In [1]:
import os
import pandas as pd
import glob

In [11]:
# Set this to the folder where you downloaded 'prokka_plasmids'
base_dir = 'annotations/prokka_plasmids/'

# 1. Find all .tsv files
# Structure: prokka_plasmids / GCA_... / plasmid_... / *.tsv
search_pattern = os.path.join(base_dir, "*", "*", "*.tsv")
tsv_files = glob.glob(search_pattern)

print(f"Found {len(tsv_files)} annotation files.")

all_genes = []

for f in tsv_files:
    try:
        # Extract metadata from the file path
        # Assuming path ends in: .../GCA_XXXX/plasmid_YYYY/GCA_XXXX_plasmid_YYYY.tsv
        parts = f.split(os.sep)
        sample_id = parts[-3]       # GCA_ folder
        plasmid_name = parts[-2]    # plasmid_ folder
        
        # Read the TSV
        df = pd.read_csv(f, sep='\t')
        
        # Filter for relevant columns to keep file size down
        # 'locus_tag', 'ftype', 'gene', 'product' are the most important
        df = df[['locus_tag', 'ftype', 'gene', 'product']]
        
        # Keep only CDS (protein coding genes) and maybe rRNA/tRNA
        df = df[df['ftype'] == 'CDS']
        
        # Add identifiers
        df['Sample'] = sample_id
        df['Plasmid_ID'] = plasmid_name
        
        all_genes.append(df)
        
    except Exception as e:
        print(f"Error parsing {f}: {e}")

# Combine into one master dataframe
master_gene_table = pd.concat(all_genes, ignore_index=True)

# 2. Quick Search for "Interesting" Genes
# Define keywords for stress/adaptation
keywords = ['heat', 'cold', 'osmotic', 'betaine', 'proline', 'arsenic', 'copper', 'mercury', 'resistance', 'toxin']

# Create a 'flag' column if the product contains a keyword
pattern = '|'.join(keywords)
master_gene_table['is_interesting'] = master_gene_table['product'].str.contains(pattern, case=False, na=False)

# Save to CSV
master_gene_table.to_csv('plasmid_gene_inventory.csv', index=False)

print("✅ Parsed all plasmids. Saved to 'plasmid_gene_inventory.csv'.")
print("\nPreview of potentially interesting genes:")

Found 88 annotation files.
✅ Parsed all plasmids. Saved to 'plasmid_gene_inventory.csv'.

Preview of potentially interesting genes:
